# Compute transfer path

### Setup Robot Simulation connection

In [63]:
%matplotlib tk
import numpy as np
import matplotlib.pyplot as plt

In [64]:
import sys
from pathlib import Path
from math import cos, sin, pi, radians

# Make sure ur3e-control is on the path
sys.path.insert(0, str(Path.cwd().parents[1]))

from my_simulation.iscoin_sim.kinematics import (
    analytical_ik,
    forward_kinematics_matrix,
    pose_to_matrix,
    matrix_to_tcp6d,
)

from my_simulation.iscoin_sim import ISCoinSim as ISCoin
from URBasic import Joint6D, TCP6D
from URBasic.waypoint6d import TCP6DDescriptor


# # Connect to the simulator
iscoin = ISCoin()
robot = iscoin.robot_control
print("Connected to simulator")

ISCoinSim connected to container 'iscoin_simulator'
Connected to simulator


Setup trimesh

In [65]:
import trimesh
print(f"trimesh version: {trimesh.__version__}")

trimesh version: 4.11.2


In [66]:
mesh_path = "../3d_objects/trapez.stl"

In [67]:
MESH_SCALE = 0.001  # STL units → meters (120 units = 0.12 m = 12 cm)

obstacle = trimesh.load(mesh_path)
print(f"Loaded: {mesh_path}")
print(f"Type: {type(obstacle)}")
print(f"Vertices: {len(obstacle.vertices)}")
print(f"Faces:    {len(obstacle.faces)}")

obstacle.apply_scale(MESH_SCALE)

# Derived sizing for plots
arrow_len = 2.0 * MESH_SCALE    # normal arrow length
text_offset = 0.3 * MESH_SCALE  # label offset from point

print(f"Watertight: {obstacle.is_watertight}")

if not obstacle.is_watertight:
    print("Mesh is not watertight — attempting repair...")
    trimesh.repair.fix_winding(obstacle)
    trimesh.repair.fill_holes(obstacle)
    trimesh.repair.fix_normals(obstacle)
    print(f"Watertight after repair: {obstacle.is_watertight}")

if not obstacle.is_watertight:
    print("Repair failed — falling back to convex hull")
    obstacle = obstacle.convex_hull
    print(f"Watertight (convex hull): {obstacle.is_watertight}")

bbox_min, bbox_max = obstacle.bounds
print(f"Bounding box min: {bbox_min}")
print(f"Bounding box max: {bbox_max}")
print(f"Size: {bbox_max - bbox_min}")
print(f"arrow_len: {arrow_len}, text_offset: {text_offset}")

Loaded: ../3d_objects/trapez.stl
Type: <class 'trimesh.base.Trimesh'>
Vertices: 12
Faces:    20
Watertight: True
Bounding box min: [-0.06 -0.04 -0.01]
Bounding box max: [0.06       0.04       0.03575261]
Size: [0.12       0.08       0.04575261]
arrow_len: 0.002, text_offset: 0.0003


trimesh custom functions

In [68]:
def is_inside(mesh, point):
    """Return True if point is inside the mesh."""
    return mesh.contains(np.array([point]))[0]

In [69]:
def segment_collides(A, B, mesh):
    """Check if straight-line segment A->B collides with the mesh.
    Collision = any point on the segment is inside or crosses the mesh surface.
    """
    # If either endpoint is inside, it's a collision
    if mesh.contains(np.array([A, B])).any():
        return True
    # Check if the segment crosses the mesh surface
    direction = B - A
    length = np.linalg.norm(direction)
    if length < 1e-10:
        return False
    d_unit = direction / length
    locations, _, _ = mesh.ray.intersects_location(
        ray_origins=np.array([A]), ray_directions=np.array([d_unit])
    )
    if len(locations) == 0:
        return False
    return bool(np.any(np.linalg.norm(locations - A, axis=1) <= length))

print("segment_collides() defined")

segment_collides() defined


TCP Initialization ( direct by default or using Cedric TCP if it works )

In [70]:
pen_length = 0.2385  # meters
robot.set_tcp(TCP6D.createFromMetersRadians(0, 0, pen_length, 0, 0, 0))

Pathing initialization

In [71]:
# Go to home position
home = Joint6D.createFromRadians(1.1859, -1.4452, 1.2389, -1.3639, -1.5693, -0.3849)
robot.movej(home, a=radians(80), v=radians(60))

tcp_home = robot.get_actual_tcp_pose()
print(f"Home TCP: {tcp_home}")

movej sent (duration=3s)
Movement OK — target reached
Home TCP: TCP6D([0.00014259111527270356, -0.35000122244420095, 0.10650296238601498, -5.06644734472236e-06, 3.139981903416153, -1.4319117020588957e-05])


In [72]:
sim_ref_point = TCP6D.createFromMetersRadians(
      0.0,      # x
     -0.20,     # y
      0.05,     # z
      0.0,      # rx
      3.14,      # ry
      0.0       # rz
  )

In [73]:
import numpy as np

drawing_z = 0.0014
safe_z = 0.13

rx, ry, rz = 0.0, 3.14, 0.0

surface_tilt_y = radians(45)  # radians — e.g. radians(45) to tilt 45° around Y

draw_speed = 0.05   # m/s — slow for drawing
travel_speed = 0.1   # m/s — faster for travel moves

In [75]:
#### FOR SIMULATION

ref_drawing_point = TCP6D.createFromMetersRadians(0, -0.20, safe_z, rx, ry, rz)

Set reference drawing point coordinate matrice

In [79]:
# Translation: map local (0, 0, z_min) to robot world (0, -0.35, 0)
z_bottom = obstacle.bounds[0][2]  # ≈ -1.0

T_trans = np.eye(4)
T_trans[0, 3] = 0.0
T_trans[1, 3] = -0.2
T_trans[2, 3] = -z_bottom  # +1.0 so that local z=-1 → world z=0

print("T_trans =")
print(T_trans)

T_trans =
[[ 1.    0.    0.    0.  ]
 [ 0.    1.    0.   -0.2 ]
 [ 0.    0.    1.    0.01]
 [ 0.    0.    0.    1.  ]]


In [80]:
# Compose: rotate local coords first, then translate to world
T_surface = T_trans # @ R_tilt add rotation matric ONLY if needed
print(f"Surface origin: x={ref_drawing_point.x:.4f}, y={ref_drawing_point.y:.4f}, tilt_y={surface_tilt_y:.2f} rad")
print()
print("T_surface =")
print(T_surface)

Surface origin: x=0.0000, y=-0.2000, tilt_y=0.79 rad

T_surface =
[[ 1.    0.    0.    0.  ]
 [ 0.    1.    0.   -0.2 ]
 [ 0.    0.    1.    0.01]
 [ 0.    0.    0.    1.  ]]


Custom library for transforming coordinates

In [81]:
def local_to_world(T_surface, lx, ly, lz=0.0):
    """Transform a local 2D point to a world TCP using surface transform."""
    world = T_surface @ np.array([lx, ly, lz, 1.0])
    return TCP6D.createFromMetersRadians(world[0], world[1], world[2], rx, ry + surface_tilt_y, rz)

Shape vizualiation ( optional )

In [82]:
# fig = plt.figure(figsize=(10, 8))
# ax = fig.add_subplot(111, projection='3d')
#
# # Draw obstacle mesh
# ax.plot_trisurf(
#     obstacle.vertices[:, 0], obstacle.vertices[:, 1], obstacle.vertices[:, 2],
#     triangles=obstacle.faces, alpha=0.15, color='gray', edgecolor='gray', linewidth=0.1
# )
#
# ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Z (m)')
# ax.set_title('Obstacle mesh (local coordinates)')
#
# # Equal scale on all axes, z starts at 0
# mid = (bbox_max + bbox_min) / 2
# half = (bbox_max - bbox_min).max() / 2
# ax.set_xlim(mid[0] - half, mid[0] + half)
# ax.set_ylim(mid[1] - half, mid[1] + half)
# ax.set_zlim(0, 2 * half)
# ax.set_box_aspect([1, 1, 1])
#
# plt.tight_layout()
# plt.show()

In [83]:
def normal_to_tcp_rotation(normal):
    """Convert a face normal to (rx, ry, rz) axis-angle for the robot.
    Pen points opposite to the normal (into the surface).
    Default (normal up = [0,0,1]) → (0, pi, 0). rz is always 0."""
    n = np.array(normal, dtype=float)
    n = n / np.linalg.norm(n)
    target = -n  # pen points into the surface

    z = np.array([0.0, 0.0, 1.0])
    c = np.dot(z, target)

    if c > 1 - 1e-10:   # normal points down → pen points up
        return 0.0, 0.0, 0.0
    if c < -1 + 1e-10:  # normal points up → pen points down (default)
        return 0.0, np.pi, 0.0

    # cross([0,0,1], target) always has z=0, so rz=0 naturally
    axis = np.cross(z, target)
    axis = axis / np.linalg.norm(axis)
    angle = np.arccos(np.clip(c, -1, 1))
    rv = axis * angle
    return rv[0], rv[1], rv[2]

# Quick test with default (normal pointing up)
print("normal [0,0,1] →", normal_to_tcp_rotation([0, 0, 1]))
print("normal [0,1,0] →", normal_to_tcp_rotation([0, 1, 0]))
print("normal [0,-1,0] →", normal_to_tcp_rotation([0, -1, 0]))

normal [0,0,1] → (0.0, 3.141592653589793, 0.0)
normal [0,1,0] → (np.float64(1.5707963267948966), np.float64(0.0), np.float64(0.0))
normal [0,-1,0] → (np.float64(-1.5707963267948966), np.float64(0.0), np.float64(0.0))


In [84]:
# Test: local normal vectors → rx, ry, rz
test_normals = {
    "straight up":        [0, 0, 1],
    "tilted +Y 45°":      [0, 1, 1],
    "tilted -Y 45°":      [0, -1, 1],
    "tilted +X 45°":      [1, 0, 1],
    "tilted -X 45°":      [-1, 0, 1],
    "horizontal +Y":      [0, 1, 0],
    "horizontal -Y":      [0, -1, 0],
    "horizontal +X":      [1, 0, 0],
    "horizontal -X":      [-1, 0, 0],
    "straight down":      [0, 0, -1],
}

print(f"{'Label':>20} | {'Normal vector':>24} | {'rx':>8} {'ry':>8} {'rz':>8}")
print("-" * 75)

for label, n in test_normals.items():
    rx, ry, rz = normal_to_tcp_rotation(n)
    n_norm = np.array(n, dtype=float)
    n_norm = n_norm / np.linalg.norm(n_norm)
    print(f"{label:>20} | [{n_norm[0]:+6.3f}, {n_norm[1]:+6.3f}, {n_norm[2]:+6.3f}] | {rx:+8.4f} {ry:+8.4f} {rz:+8.4f}")

               Label |            Normal vector |       rx       ry       rz
---------------------------------------------------------------------------
         straight up | [+0.000, +0.000, +1.000] |  +0.0000  +3.1416  +0.0000
       tilted +Y 45° | [+0.000, +0.707, +0.707] |  +2.3562  +0.0000  +0.0000
       tilted -Y 45° | [+0.000, -0.707, +0.707] |  -2.3562  +0.0000  +0.0000
       tilted +X 45° | [+0.707, +0.000, +0.707] |  +0.0000  -2.3562  +0.0000
       tilted -X 45° | [-0.707, +0.000, +0.707] |  +0.0000  +2.3562  -0.0000
       horizontal +Y | [+0.000, +1.000, +0.000] |  +1.5708  +0.0000  +0.0000
       horizontal -Y | [+0.000, -1.000, +0.000] |  -1.5708  +0.0000  +0.0000
       horizontal +X | [+1.000, +0.000, +0.000] |  +0.0000  -1.5708  +0.0000
       horizontal -X | [-1.000, +0.000, +0.000] |  +0.0000  +1.5708  -0.0000
       straight down | [+0.000, +0.000, -1.000] |  +0.0000  +0.0000  +0.0000


In [85]:
# Two points at mid-height, on opposite y sides just outside the obstacle
y_margin = 0.01  # 1cm outside the mesh boundary
p1_local = np.array([0.0, bbox_min[1] - y_margin, (bbox_min[2] + bbox_max[2]) / 2])
p2_local = np.array([0.0, bbox_max[1] + y_margin, (bbox_min[2] + bbox_max[2]) / 2])

# Find the closest face on the mesh to derive the surface normal
_, _, face_idx = trimesh.proximity.closest_point(obstacle, np.array([p1_local, p2_local]))
n1 = obstacle.face_normals[face_idx[0]]
n2 = obstacle.face_normals[face_idx[1]]

# p1 and p2 in local coordinates with face normals → rx, ry, rz
rx1, ry1, rz1 = normal_to_tcp_rotation(n1)
rx2, ry2, rz2 = normal_to_tcp_rotation(n2)

p1 = TCP6D.createFromMetersRadians(p1_local[0], p1_local[1], p1_local[2], rx1, ry1, rz1)
p2 = TCP6D.createFromMetersRadians(p2_local[0], p2_local[1], p2_local[2], rx2, ry2, rz2)

print("=== Local coordinates ===")
print(f"p1: x={p1.x:.4f}, y={p1.y:.4f}, z={p1.z:.4f}, rx={rx1:.4f}, ry={ry1:.4f}, rz={rz1:.4f}")
print(f"    normal: {n1}")
print(f"p2: x={p2.x:.4f}, y={p2.y:.4f}, z={p2.z:.4f}, rx={rx2:.4f}, ry={ry2:.4f}, rz={rz2:.4f}")
print(f"    normal: {n2}")

=== Local coordinates ===
p1: x=0.0000, y=-0.0500, z=0.0129, rx=-2.0944, ry=0.0000, rz=0.0000
    normal: [ 0.         -0.86602536  0.50000007]
p2: x=0.0000, y=0.0500, z=0.0129, rx=2.0944, ry=0.0000, rz=0.0000
    normal: [0.         0.86602544 0.49999994]


In [86]:
# Convert p1 and p2 to world (robot) coordinates
w1 = T_surface @ np.array([p1_local[0], p1_local[1], p1_local[2], 1.0])
w2 = T_surface @ np.array([p2_local[0], p2_local[1], p2_local[2], 1.0])

p1_world = TCP6D.createFromMetersRadians(w1[0], w1[1], w1[2], rx1, ry1, rz1)
p2_world = TCP6D.createFromMetersRadians(w2[0], w2[1], w2[2], rx2, ry2, rz2)

print("=== World (robot) coordinates ===")
print(f"p1_world: x={p1_world.x:.4f}, y={p1_world.y:.4f}, z={p1_world.z:.4f}, rx={rx1:.4f}, ry={ry1:.4f}, rz={rz1:.4f}")
print(f"p2_world: x={p2_world.x:.4f}, y={p2_world.y:.4f}, z={p2_world.z:.4f}, rx={rx2:.4f}, ry={ry2:.4f}, rz={rz2:.4f}")

=== World (robot) coordinates ===
p1_world: x=0.0000, y=-0.2500, z=0.0229, rx=-2.0944, ry=0.0000, rz=0.0000
p2_world: x=0.0000, y=-0.1500, z=0.0229, rx=2.0944, ry=0.0000, rz=0.0000


In [89]:
from matplotlib.widgets import Slider

fig = plt.figure(figsize=(12, 9))
fig.subplots_adjust(bottom=0.20)
ax = fig.add_subplot(111, projection='3d')

# Draw obstacle mesh
ax.plot_trisurf(
    obstacle.vertices[:, 0], obstacle.vertices[:, 1], obstacle.vertices[:, 2],
    triangles=obstacle.faces, alpha=0.15, color='gray', edgecolor='gray', linewidth=0.1
)

# Draw p1 and p2 with their face normals
for label, pt, normal in [('p1', p1_local, n1), ('p2', p2_local, n2)]:
    ax.scatter(*pt, color='red', s=60, zorder=5)
    ax.text(pt[0]+text_offset, pt[1]+text_offset, pt[2]+text_offset, label, fontsize=10, color='red')
    ax.quiver(pt[0], pt[1], pt[2],
              normal[0]*arrow_len, normal[1]*arrow_len, normal[2]*arrow_len,
              color='blue', arrow_length_ratio=0.15, linewidth=2)

ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Z (m)')
ax.set_title('Obstacle mesh with p1, p2 and face normals')

# Equal scale on all axes, z starts at 0
all_pts = np.vstack([obstacle.vertices, p1_local, p2_local])
lo, hi = all_pts.min(axis=0), all_pts.max(axis=0)
mid = (hi + lo) / 2
half = (hi - lo).max() / 2 * 1.2
ax.set_xlim(mid[0] - half, mid[0] + half)
ax.set_ylim(mid[1] - half, mid[1] + half)
ax.set_zlim(0, 2 * half)
ax.set_box_aspect([1, 1, 1])

# Lock mouse rotation, use sliders instead
ax.view_init(elev=25, azim=-60)
ax.disable_mouse_rotation()

azim_ax = fig.add_axes([0.2, 0.08, 0.6, 0.03])
azim_slider = Slider(azim_ax, 'Azimuth', -180, 180, valinit=-60, valstep=1)

elev_ax = fig.add_axes([0.2, 0.03, 0.6, 0.03])
elev_slider = Slider(elev_ax, 'Elevation', -90, 90, valinit=25, valstep=1)

def update_view(val):
    ax.view_init(elev=elev_slider.val, azim=azim_slider.val)
    fig.canvas.draw_idle()

azim_slider.on_changed(update_view)
elev_slider.on_changed(update_view)

plt.show()

In [88]:
from my_simulation.iscoin_sim.kinematics import select_closest_ik

# Get current joint positions (home) as reference
q_home = np.array(home.toList())

# Compute IK for both world points
for label, p_world in [('p1_world', p1_world), ('p2_world', p2_world)]:
    T_desired = pose_to_matrix(p_world)
    T_flange = T_desired @ np.linalg.inv(robot._tcp_offset)
    solutions = analytical_ik(T_flange)

    print(f"\n=== IK solutions for {label}: {p_world} ===")
    if not solutions:
        print("  No solutions found!")
        continue

    print(f"  Found {len(solutions)} solutions:")
    for i, sol in enumerate(solutions):
        dist = np.linalg.norm(sol - q_home)
        degs = np.degrees(sol)
        print(f"  [{i}] {np.array2string(degs, precision=2, suppress_small=True)}° (dist={dist:.4f})")

    best = select_closest_ik(solutions, q_home)
    best_idx = next(i for i, s in enumerate(solutions) if np.allclose(s, best))
    print(f"\n  -> Best (closest to home): [{best_idx}] {np.array2string(np.degrees(best), precision=2, suppress_small=True)}°")

    if label == 'p1_world':
        j1 = Joint6D.createFromRadians(*best.tolist())
    else:
        j2 = Joint6D.createFromRadians(*best.tolist())

print(f"\nj1 = {j1}")
print(f"j2 = {j2}")


=== IK solutions for p1_world: TCP6D([0.0, -0.25, 0.02287630605697632, -2.0943951867313157, 0.0, 0.0]) ===
  No solutions found!

=== IK solutions for p2_world: TCP6D([0.0, -0.15000000000000002, 0.02287630605697632, 2.094395033557966, 0.0, 0.0]) ===
  Found 8 solutions:
  [0] [-164.03  -60.89  151.95   24.41  146.37  119.78]° (dist=6.6989)
  [1] [-164.03   61.27 -151.95 -153.85  146.37  119.78]° (dist=7.9206)
  [2] [-164.03 -177.57  160.46  -47.42 -146.37  -60.22]° (dist=4.8239)
  [3] [-164.03  -59.32 -160.46  155.25 -146.37  -60.22]° (dist=7.1336)
  [4] [ 164.03 -120.68  160.46   24.75  146.37   60.22]° (dist=5.2898)
  [5] [ 164.03   -2.43 -160.46 -132.58  146.37   60.22]° (dist=6.4085)
  [6] [ 164.03  118.73  151.95  -26.15 -146.37 -119.78]° (dist=4.6781)
  [7] [ 164.03 -119.11 -151.95  155.59 -146.37 -119.78]° (dist=6.2350)

  -> Best (closest to home): [6] [ 164.03  118.73  151.95  -26.15 -146.37 -119.78]°


NameError: name 'j1' is not defined

In [90]:
# === IK Reachability Visualization on Mesh Surface ===
# Sample 10 random points per face, check IK reachability, and plot results.

from matplotlib.widgets import Slider

# --- 1. Sample 10 points per face using barycentric coordinates ---
rng = np.random.default_rng(42)
N_PER_FACE = 10

sampled_points = []
sampled_face_indices = []

for face_idx in range(len(obstacle.faces)):
    verts = obstacle.vertices[obstacle.faces[face_idx]]  # (3, 3)
    # Random barycentric coordinates
    r1 = rng.random(N_PER_FACE)
    r2 = rng.random(N_PER_FACE)
    sqrt_r1 = np.sqrt(r1)
    u = 1 - sqrt_r1
    v = sqrt_r1 * (1 - r2)
    w = sqrt_r1 * r2
    pts = u[:, None] * verts[0] + v[:, None] * verts[1] + w[:, None] * verts[2]
    sampled_points.append(pts)
    sampled_face_indices.extend([face_idx] * N_PER_FACE)

sampled_points = np.vstack(sampled_points)
sampled_face_indices = np.array(sampled_face_indices)

print(f"Sampled {len(sampled_points)} points ({N_PER_FACE} per face, {len(obstacle.faces)} faces)")

# --- 2-4. Compute IK reachability for each sampled point ---
reachable_mask = np.zeros(len(sampled_points), dtype=bool)

for i, (pt, fi) in enumerate(zip(sampled_points, sampled_face_indices)):
    # Get face normal
    normal = obstacle.face_normals[fi]

    # Convert to world TCP pose
    world_pos = T_surface @ np.array([pt[0], pt[1], pt[2], 1.0])
    rx_i, ry_i, rz_i = normal_to_tcp_rotation(normal)
    tcp_world = TCP6D.createFromMetersRadians(
        world_pos[0], world_pos[1], world_pos[2], rx_i, ry_i, rz_i
    )

    # Compute IK
    T_desired = pose_to_matrix(tcp_world)
    T_flange = T_desired @ np.linalg.inv(robot._tcp_offset)
    solutions = analytical_ik(T_flange)
    reachable_mask[i] = len(solutions) > 0

n_reachable = reachable_mask.sum()
n_total = len(sampled_points)
print(f"{n_reachable}/{n_total} points reachable ({100 * n_reachable / n_total:.1f}%)")

# --- 5. Plot ---
reachable_pts = sampled_points[reachable_mask]
unreachable_pts = sampled_points[~reachable_mask]

fig = plt.figure(figsize=(12, 9))
fig.subplots_adjust(bottom=0.20)
ax = fig.add_subplot(111, projection='3d')

# Obstacle mesh in gray
ax.plot_trisurf(
    obstacle.vertices[:, 0], obstacle.vertices[:, 1], obstacle.vertices[:, 2],
    triangles=obstacle.faces, alpha=0.15, color='gray', edgecolor='gray', linewidth=0.1
)

# Green dots = reachable, Red dots = unreachable
if len(reachable_pts) > 0:
    ax.scatter(reachable_pts[:, 0], reachable_pts[:, 1], reachable_pts[:, 2],
               color='green', s=20, zorder=5, label=f'Reachable ({len(reachable_pts)})')
if len(unreachable_pts) > 0:
    ax.scatter(unreachable_pts[:, 0], unreachable_pts[:, 1], unreachable_pts[:, 2],
               color='red', s=20, zorder=5, label=f'Unreachable ({len(unreachable_pts)})')

ax.legend(loc='upper left')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Z (m)')
ax.set_title(f'IK Reachability on Mesh Surface — {n_reachable}/{n_total} reachable ({100*n_reachable/n_total:.1f}%)')

# Equal scale, z starts at 0
mid = (bbox_max + bbox_min) / 2
half = (bbox_max - bbox_min).max() / 2 * 1.2
ax.set_xlim(mid[0] - half, mid[0] + half)
ax.set_ylim(mid[1] - half, mid[1] + half)
ax.set_zlim(0, 2 * half)
ax.set_box_aspect([1, 1, 1])

# Sliders for rotation, mouse rotation locked
ax.view_init(elev=25, azim=-60)
ax.disable_mouse_rotation()

azim_ax = fig.add_axes([0.2, 0.08, 0.6, 0.03])
azim_slider = Slider(azim_ax, 'Azimuth', -180, 180, valinit=-60, valstep=1)

elev_ax = fig.add_axes([0.2, 0.03, 0.6, 0.03])
elev_slider = Slider(elev_ax, 'Elevation', -90, 90, valinit=25, valstep=1)

def update_view(val):
    ax.view_init(elev=elev_slider.val, azim=azim_slider.val)
    fig.canvas.draw_idle()

azim_slider.on_changed(update_view)
elev_slider.on_changed(update_view)

plt.show()

Sampled 200 points (10 per face, 20 faces)
121/200 points reachable (60.5%)


In [91]:
# === IK Reachability with Cone Tolerance ===
# Green  = reachable at exact normal orientation
# Blue   = reachable with a perturbed orientation (within ±cone angle)
# Red    = unreachable even with perturbations
# Normals pointing toward -z are always rejected.

from matplotlib.widgets import Slider
from scipy.spatial.transform import Rotation

CONE_HALF_ANGLE = np.radians(15)  # ±15° around the face normal
N_CONE_SAMPLES = 20               # random orientations to try per point

def sample_cone_normals(normal, half_angle, n_samples, rng):
    """Sample n_samples directions within a cone of half_angle around normal.
    Returns (n_samples, 3) array of unit vectors."""
    n = np.array(normal, dtype=float)
    n = n / np.linalg.norm(n)

    # Build a local frame where n is the z-axis
    if abs(n[2]) < 0.99:
        t = np.cross(n, [0, 0, 1])
    else:
        t = np.cross(n, [1, 0, 0])
    t = t / np.linalg.norm(t)
    b = np.cross(n, t)

    # Sample uniformly within the cone
    # cos(theta) uniform in [cos(half_angle), 1]
    cos_theta = rng.uniform(np.cos(half_angle), 1.0, size=n_samples)
    sin_theta = np.sqrt(1 - cos_theta**2)
    phi = rng.uniform(0, 2 * np.pi, size=n_samples)

    # Directions in local frame, then rotate to world
    dirs = (sin_theta * np.cos(phi))[:, None] * t \
         + (sin_theta * np.sin(phi))[:, None] * b \
         + cos_theta[:, None] * n
    return dirs

def check_ik_for_pose(pt_local, rx_i, ry_i, rz_i):
    """Convert local point + orientation to world pose and check IK. Returns True if reachable."""
    world_pos = T_surface @ np.array([pt_local[0], pt_local[1], pt_local[2], 1.0])
    tcp_world = TCP6D.createFromMetersRadians(
        world_pos[0], world_pos[1], world_pos[2], rx_i, ry_i, rz_i
    )
    T_desired = pose_to_matrix(tcp_world)
    T_flange = T_desired @ np.linalg.inv(robot._tcp_offset)
    solutions = analytical_ik(T_flange)
    return len(solutions) > 0

# --- Classify each sampled point ---
# 0 = unreachable, 1 = reachable at exact normal, 2 = reachable with perturbation
classification = np.zeros(len(sampled_points), dtype=int)

rng_cone = np.random.default_rng(123)

for i, (pt, fi) in enumerate(zip(sampled_points, sampled_face_indices)):
    normal = obstacle.face_normals[fi]

    # Skip faces whose normal points toward -z
    if normal[2] < 0:
        classification[i] = 0
        continue

    # Try exact normal first
    rx_i, ry_i, rz_i = normal_to_tcp_rotation(normal)
    if check_ik_for_pose(pt, rx_i, ry_i, rz_i):
        classification[i] = 1
        continue

    # Try perturbed orientations within the cone
    cone_dirs = sample_cone_normals(normal, CONE_HALF_ANGLE, N_CONE_SAMPLES, rng_cone)
    found = False
    for d in cone_dirs:
        # Reject any direction pointing toward -z
        if d[2] < 0:
            continue
        rx_c, ry_c, rz_c = normal_to_tcp_rotation(d)
        if check_ik_for_pose(pt, rx_c, ry_c, rz_c):
            found = True
            break
    classification[i] = 2 if found else 0

n_exact = (classification == 1).sum()
n_cone = (classification == 2).sum()
n_unreach = (classification == 0).sum()
n_total = len(classification)

print(f"Exact normal reachable (green): {n_exact}/{n_total}")
print(f"Cone-perturbed reachable (blue): {n_cone}/{n_total}")
print(f"Unreachable (red):               {n_unreach}/{n_total}")
print(f"Total reachable: {n_exact + n_cone}/{n_total} ({100*(n_exact+n_cone)/n_total:.1f}%)")

# --- Plot ---
exact_pts = sampled_points[classification == 1]
cone_pts = sampled_points[classification == 2]
unreach_pts = sampled_points[classification == 0]

fig = plt.figure(figsize=(12, 9))
fig.subplots_adjust(bottom=0.20)
ax = fig.add_subplot(111, projection='3d')

ax.plot_trisurf(
    obstacle.vertices[:, 0], obstacle.vertices[:, 1], obstacle.vertices[:, 2],
    triangles=obstacle.faces, alpha=0.15, color='gray', edgecolor='gray', linewidth=0.1
)

if len(exact_pts) > 0:
    ax.scatter(exact_pts[:, 0], exact_pts[:, 1], exact_pts[:, 2],
               color='green', s=20, zorder=5, label=f'Exact normal ({len(exact_pts)})')
if len(cone_pts) > 0:
    ax.scatter(cone_pts[:, 0], cone_pts[:, 1], cone_pts[:, 2],
               color='blue', s=20, zorder=5, label=f'Cone ±{int(np.degrees(CONE_HALF_ANGLE))}° ({len(cone_pts)})')
if len(unreach_pts) > 0:
    ax.scatter(unreach_pts[:, 0], unreach_pts[:, 1], unreach_pts[:, 2],
               color='red', s=20, zorder=5, label=f'Unreachable ({len(unreach_pts)})')

ax.legend(loc='upper left')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Z (m)')
ax.set_title(f'IK Reachability with ±{int(np.degrees(CONE_HALF_ANGLE))}° cone — '
             f'{n_exact} exact + {n_cone} cone / {n_total} total')

mid = (bbox_max + bbox_min) / 2
half = (bbox_max - bbox_min).max() / 2 * 1.2
ax.set_xlim(mid[0] - half, mid[0] + half)
ax.set_ylim(mid[1] - half, mid[1] + half)
ax.set_zlim(0, 2 * half)
ax.set_box_aspect([1, 1, 1])

ax.view_init(elev=25, azim=-60)
ax.disable_mouse_rotation()

azim_ax = fig.add_axes([0.2, 0.08, 0.6, 0.03])
azim_slider = Slider(azim_ax, 'Azimuth', -180, 180, valinit=-60, valstep=1)

elev_ax = fig.add_axes([0.2, 0.03, 0.6, 0.03])
elev_slider = Slider(elev_ax, 'Elevation', -90, 90, valinit=25, valstep=1)

def update_view(val):
    ax.view_init(elev=elev_slider.val, azim=azim_slider.val)
    fig.canvas.draw_idle()

azim_slider.on_changed(update_view)
elev_slider.on_changed(update_view)

plt.show()

Exact normal reachable (green): 121/200
Cone-perturbed reachable (blue): 39/200
Unreachable (red):               40/200
Total reachable: 160/200 (80.0%)
